# Classification Automatique des Articles de Blog Weeb

Ce notebook présente la démarche pas-à-pas pour concevoir, entraîner et évaluer le modèle de Machine Learning utilisé par l'application **Weeb** pour classifier automatiquement ses articles dans l'une des 4 catégories cibles :
- **Tech** : Articles sur le développement, le code, Git, Docker, etc.
- **Design** : Articles sur l'UI/UX, Figma, la typographie, le responsive, etc.
- **Marketing** : Articles sur le SEO, le copywriting, la croissance, les campagnes e-mails.
- **Business** : Articles sur le SaaS, l'entrepreneuriat, la levée de fonds, l'agilité.

## 1. Importation des bibliothèques nécessaires

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import joblib

## 2. Chargement et exploration des données

Nous chargeons le jeu de données synthétique généré pour le projet (`dataset.csv`).

In [ ]:
# Chargement du fichier
df = pd.read_csv('dataset.csv')
print(f"Taille du dataset : {df.shape[0]} lignes, {df.shape[1]} colonnes.")
df.head()

### Visualisation de la distribution des classes

In [ ]:
class_counts = df['category'].value_counts()
print(class_counts)

plt.figure(figsize=(8, 4))
class_counts.plot(kind='bar', color=['#9333EA', '#3B82F6', '#10B981', '#F59E0B'])
plt.title("Distribution des articles par catégorie")
plt.xlabel("Catégories")
plt.ylabel("Nombre d'articles")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 3. Préparation du texte et division Train/Test

Pour maximiser les performances, nous concaténons le titre, l'extrait et le contenu de chaque article. Ensuite, nous divisons notre jeu de données en sous-ensembles d'entraînement (80%) et de test (20%).

In [ ]:
# Combinaison des colonnes textuelles
df['full_text'] = df['title'] + " " + df['excerpt'] + " " + df['content']

X = df['full_text']
y = df['category']

# Split stratifié pour conserver la proportion de chaque catégorie dans les jeux de train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print(f"Taille du jeu d'entraînement : {len(X_train)} exemples")
print(f"Taille du jeu de test : {len(X_test)} exemples")

## 4. Création et entraînement du Pipeline

Nous utilisons un pipeline `scikit-learn` comprenant :
1. **TF-IDF Vectorizer** : Transforme le texte brut en vecteurs de fréquences numériques de mots (en ignorant les stop-words anglais).
2. **Logistic Regression** : Un classificateur linéaire robuste et rapide pour la classification de textes.

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        stop_words='english',
        max_df=0.95,
        min_df=2,
        ngram_range=(1, 2)
    )),
    ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=42))
])

# Entraînement
pipeline.fit(X_train, y_train)
print("Entraînement terminé !")

## 5. Évaluation des performances

Nous mesurons l'exactitude globale et générons un rapport détaillé avec la précision, le rappel et le score F1.

In [ ]:
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Exactitude globale : {accuracy * 100:.2f}%\n")

print("Rapport de classification :");
print(classification_report(y_test, y_pred))

### Matrice de confusion

In [ ]:
labels = sorted(list(y.unique()))
cm = confusion_matrix(y_test, y_pred, labels=labels)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap=plt.cm.Purples)
plt.title("Matrice de Confusion du Classificateur Weeb")
plt.xticks(rotation=45)
plt.show()

## 6. Sauvegarde du modèle

Nous exportons le pipeline complet pour l'utiliser dans notre API REST Django.

In [ ]:
joblib.dump(pipeline, 'model.joblib')
print("Modèle model.joblib enregistré !")

## 7. Exemple de prédiction en conditions réelles

In [ ]:
nouveau_texte = "Mastering UI layouts and spacing in Figma. Let's talk about contrast and user interfaces."
pred = pipeline.predict([nouveau_texte])
prob = pipeline.predict_proba([nouveau_texte])

print(f"Texte : '{nouveau_texte}'")
print(f"Catégorie prédite : {pred[0]}")
print(f"Probabilités pour chaque classe :")
for label, pr in zip(labels, prob[0]):
    print(f"- {label} : {pr * 100:.2f}%")